# Step 02 - Reliability Tables Validation

Reproduce Tables 2/3 and the Bland-Altman/ICC/CCC statistics in R, then compare against the Python-generated references in `B_Revised_Figures/`.

In [ ]:
source("../R/00_config.R")
source("../R/01_load_data.R")
source("../R/03_reliability_tables.R")
source("../R/99_validate_csv.R")

In [ ]:
config <- analysis_config()
ensure_dir(config$output_dir)
required_input_paths(config)

In [ ]:
stopifnot(inputs_available(config))
df <- load_tcd_data(config)
tables <- build_tables_and_stats_r(df)

In [ ]:
outputs <- list(
  Table2_Baseline_All15_Reliability = tables$table2,
  Table3_Min1_Min2_All15_Reliability = tables$table3,
  BA_ICC_CCC_Statistics = tables$stats,
  SupplementaryTable_BA_Statistics = tables$stats
)

for (name in names(outputs)) {
  out_path <- file.path(config$output_dir, paste0(name, ".csv"))
  write_csv_minimal(outputs[[name]], out_path)
  comparison <- compare_csv_numeric(out_path, file.path(config$reference_dir, paste0(name, ".csv")), tolerance = 1e-8)
  print_csv_comparison(paste0(name, " comparison against Python reference:"), comparison)
  if (!comparison$same_values) print(head(comparison$mismatches, 12))
}